[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/13_GradientBoosting_XGBoost.ipynb#copy=true)


# Gradient Boosting and XGBoost

## Learning objectives

By the end of this lesson, you should be able to:

- contrast bagging/random forests with sequential boosting;
- explain learning rate, number of stages, and tree complexity;
- fit and evaluate a gradient-boosting classifier;
- describe XGBoost as a regularized, efficient gradient-boosting implementation.

**Student Learning Outcome:** Explain the tradeoffs of a nonlinear ensemble and evaluate it on data not used for fitting.


## Boosting: learn from previous errors

Random forests grow trees independently and vote. **Gradient boosting** grows weak trees sequentially: each new tree is chosen to reduce the loss left by the current ensemble. For classification, later trees focus on observations earlier trees have not modeled well.

The learning rate controls how much each new tree changes the ensemble. Smaller learning rates often require more stages. Deep trees or too many stages can overfit, so validation remains essential.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

RANDOM_STATE = 232
wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
class_names = wine.target_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("features:", list(X.columns))
print("training observations:", len(X_train), "test observations:", len(X_test))


## Part 1 — Fit a gradient-boosting model

`HistGradientBoostingClassifier` is a scikit-learn gradient-boosting implementation. It provides the core boosting workflow without requiring a separate package.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

boosted = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=150,
                                          max_leaf_nodes=12, l2_regularization=0.5,
                                          random_state=RANDOM_STATE)
boosted.fit(X_train, y_train)
predictions = boosted.predict(X_test)
print("training accuracy:", round(boosted.score(X_train, y_train), 3))
print("test accuracy:", round(accuracy_score(y_test, predictions), 3))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=class_names)
plt.title("Gradient-boosting test predictions")
plt.show()


### Checkpoint 1

Contrast the role of trees in a random forest with their role in gradient boosting. Why might a very large number of boosting stages be risky?

> **YOUR ANSWER:**


## Part 2 — Hyperparameters express a tradeoff

- `learning_rate` shrinks each tree's contribution.
- `max_iter` is the number of boosting stages.
- `max_leaf_nodes` limits the complexity of each stage.
- `l2_regularization` penalizes overly flexible updates.

Choose these settings with cross-validation or a validation set, not by repeatedly selecting the best final test result.


In [ ]:
rows = []
for rate, stages in [(0.03, 300), (0.08, 150), (0.2, 60)]:
    candidate = HistGradientBoostingClassifier(learning_rate=rate, max_iter=stages,
                                                max_leaf_nodes=12, l2_regularization=0.5,
                                                random_state=RANDOM_STATE).fit(X_train, y_train)
    rows.append({"learning rate": rate, "stages": stages,
                 "training accuracy": candidate.score(X_train, y_train),
                 "test accuracy": candidate.score(X_test, y_test)})
pd.DataFrame(rows).round(3)


### Practice 2

Choose two rows with similar training accuracy but different test accuracy. What does this suggest about tuning an ensemble rather than assuming that more stages are always better?

> **YOUR ANSWER:**


## Part 3 — Where XGBoost fits

**XGBoost** is a widely used implementation of gradient-boosted trees. It adds efficient computation, regularization options, and practical training tools. The statistical idea is still boosting: add regularized trees that improve the current model's loss.

The optional cell below runs only when the `xgboost` package is installed. In Colab, use `!pip -q install xgboost` in a separate setup cell if needed. The course concepts do not depend on a package-specific score.


In [ ]:
try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    xgboost_available = False

if xgboost_available:
    xgb = XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.08,
                        subsample=0.9, colsample_bytree=0.9, eval_metric="mlogloss",
                        random_state=RANDOM_STATE)
    xgb.fit(X_train, y_train)
    print("XGBoost test accuracy:", round(xgb.score(X_test, y_test), 3))
else:
    print("Optional package unavailable. The required boosting model is above.")


## Lesson summary

Boosting builds an additive model sequentially; random forests average independently grown trees. Both are nonlinear ensembles, and both must be selected and evaluated without test-set leakage. The next lesson compares nonlinear models with a linear baseline before we move to neural networks.
